# Subspace Robust Wasserstein

The subspace robust distance maximizes the projected Wasserstein distance over rank-constrained projections.  In two dimensions and rank one, this means selecting a direction
$$
\theta^\star\in\arg\max_{\|\theta\|=1} W_2\big((P_\theta)_\#\alpha,(P_\theta)_\#\beta\big).
$$
The figure shows the selected discriminating direction and the projected monotone matching that realizes the one-dimensional cost.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import ot

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "subspace-robust-wasserstein"
out = figure_dir(NAME)
rng = np.random.default_rng(64)


The point clouds are designed so that the main discrepancy lives along an oblique direction.  A small angular sweep is enough to illustrate the rank-one robust projection.


In [ ]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

n = 26
R = rot(np.deg2rad(34))
x = rng.normal(size=(n, 2)) @ np.diag([0.33, 0.15]) @ R.T + np.array([-0.18, -0.05])
y = rng.normal(size=(n, 2)) @ np.diag([0.34, 0.15]) @ R.T + np.array([0.52, 0.45])
y[:9] += 0.20 * R[:, 0]

def projected_w2(theta):
    px = np.sort(x @ theta)
    py = np.sort(y @ theta)
    return float(np.mean((px-py)**2))

angles = np.linspace(0, np.pi, 91, endpoint=False)
dirs = np.column_stack([np.cos(angles), np.sin(angles)])
vals = np.array([projected_w2(theta) for theta in dirs])
best = int(np.argmax(vals))
theta = dirs[best]
px, py = x @ theta, y @ theta
ix, iy = np.argsort(px), np.argsort(py)
print(float(angles[best]), float(vals[best]))


The second panel is purely projected: red and blue points lie on the selected line, and violet links show the sorted one-dimensional optimal matching.


In [ ]:
fig, ax = plt.subplots(figsize=(2.35, 2.25))
draw_point_clouds(ax, x, y)
for angle in np.deg2rad([5, 62, 118]):
    d = np.array([np.cos(angle), np.sin(angle)])
    line = np.vstack([-1.7*d, 1.7*d])
    ax.plot(line[:, 0], line[:, 1], color=LIGHT_GRAY, lw=0.7, alpha=0.55, zorder=0)
line = np.vstack([-1.85*theta, 1.85*theta])
ax.plot(line[:, 0], line[:, 1], color=VIOLET, lw=1.25, alpha=0.88, zorder=0)
ax.arrow(1.35*theta[0], 1.35*theta[1], 0.30*theta[0], 0.30*theta[1], width=0.010, head_width=0.075, head_length=0.10, length_includes_head=True, color=VIOLET, alpha=0.86, zorder=0)
pts = np.vstack([x, y])
ax.set_xlim(pts[:, 0].min()-0.35, pts[:, 0].max()+0.35)
ax.set_ylim(pts[:, 1].min()-0.35, pts[:, 1].max()+0.35)
ax.set_aspect("equal")
remove_axes(ax)
save_pdf(fig, out / "selected-subspace.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.85, 1.80))
base0, base1 = 0.0, 0.28
for k in range(n):
    ax.plot([px[ix[k]], py[iy[k]]], [base0, base1], color=VIOLET, lw=0.55, alpha=0.34, zorder=1)
ax.scatter(px, np.full(n, base0), s=DIRAC_MARKER_SIZE*0.70, color=RED, marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.scatter(py, np.full(n, base1), s=DIRAC_MARKER_SIZE*0.70, color=BLUE, marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.plot([min(px.min(), py.min()), max(px.max(), py.max())], [-0.13, -0.13], color=VIOLET, lw=1.0, alpha=0.78)
ax.set_xlim(min(px.min(), py.min())-0.25, max(px.max(), py.max())+0.25)
ax.set_ylim(-0.30, 0.55)
remove_axes(ax)
save_pdf(fig, out / "projected-matching.pdf", pad_inches=0.055)
plt.close(fig)
